# Land Cover Thematic Accuracy

Thematic accuracy of OpenStreetMap land cover data in comparison to the CORINE Land Cover (CLC) dataset.
This indicator can be calculated for multiple or a single CLC class(es).

### Methods and Data
- Extrinsic approach.

#### CORINE Land Cover
In its current form, the [CORINE Land Cover (CLC)](https://land.copernicus.eu/en/products/corine-land-cover) product offers a pan-European land cover and land use
inventory with 44 thematic classes, ranging from broad forested areas to individual vineyards.
CORINE uses a 3-level nomenclature for land cover classes.
Here, we use the "CORINE Land Cover 5 ha, Stand 2021 (CLC5-2021)"
provided by the German Federal Agency for Cartography and Geodesy and
level 2 of the nomenclature (e.g. 1.1 Urban Fabric, 1.2 Industrial, commercial and transport units).

### Preprocessing:

OSM features are assigned a CLC class based on their tags as specified below. 
We calculate the intersection of OSM land cover polygons and CORINE land cover polygons.
The resulting polygons contain both the OSM CLC class and the CORINE class.

| CLC (level 2)                                        | OSM tags                                                                                                                                                                                | 
|------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| 1.1 Urban Fabric                                     | `landuse in (residential, retail)`                                                                                                                                                      |
| 1.2 	Industrial, commercial and transport units      | `landuse in (industrial, commercial, port, railway, lock) or leisure in (marina)`                                                                                                       |
| 1.3 	Mine, dump and construction sites               | `landuse in (quarry, construction, landfill, brownfield)`                                                                                                                               |
| 1.4 	Artificial, non-agricultural vegetated areas    | `landuse in (recreation_ground, allotments, village_green, cemetery, grass) or leisure in (park, garden, pitch, golf_course, playground, stadium, recreation_ground, common, dog_park)` |
| 2.1 	Arable land                                     | `landuse in (greenhouse_horticulture, greenhouse, farmland, farmyard)`                                                                                                                  |
| 2.2 	Permanent crops                                 | `landuse in (vineyard, orchard)`                                                                                                                                                        |
| 2.3 	Pastures                                        | `landuse in (meadow)`                                                                                                                                                                   |
| 2.4 	Heterogeneous agricultural areas                | *For class 2.4 many of the OSM tags from 2.1 apply. Therefore class 2.4 is never assigned.*                                                                                             |
| 3.1 	Forests                                         | `landuse in (forest) or natural in (wood)`                                                                                                                                              |
| 3.2 	Scrub and/or herbaceous vegetation associations | `landuse in (greenfield) or natural in (grassland, scrub, heath, fell)`                                                                                                                 |
| 3.3 	Open spaces with little or no vegetation        | `natural in (beach, scree, shingle, bare_rock, sand, glacier, mud, glacier, rock, cliff, fill)`                                                                                         |
| 4.1 	Inland wetlands                                 | `natural in (wetland)`                                                                                                                                                                  |
| 4.2 	Maritime wetlands                               | `landuse in (salt_pond)`                                                                                                                                                                |
| 5.1 	Inland waters                                   | `natural in (water, pond) or landuse in (basin, reservoir)`                                                                                                                             |
| 5.2 	Marine waters                                   | *Marine waters are not mapped in OSM. Therefore class 5.2 is never assigned.*                                                                                                           |

#### Indicator Calculation:

For the area-of-interest we fetch and clip polygons from the preprocessing step.
Next, we create confusion matrix between OSM and CORINE CLS classes.
This is the basis for calculating precision, recall and F1 score.
These calculations consider the area / size of the land cover polygons.

### References

- Schultz, Michael, Janek Voss, Michael Auer, Sarah Carter, and Alexander Zipf. 2017. “Open Land Cover from OpenStreetMap and Remote Sensing.” International Journal of Applied Earth Observation and Geoinformation 63 (May): 206–13. https://doi.org/10.1016/j.jag.2017.07.014.



In [1]:
import geojson
import requests
import geopandas as gpd
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys

In [2]:
indicator = "/land-cover-thematic-accuracy"
topic = "land-cover"
input_geom_path = "friedberg.geojson"
output_geom_path = "your_output_layer.gpkg"
max_workers = 20

base_url = "https://api.quality.ohsome.org/v1-test"
endpoint = "/indicators"
url = base_url + endpoint + indicator

gdf = gpd.read_file(input_geom_path)
gdf["result_value"] = pd.Series([None] * len(gdf), dtype="float")
gdf["response_time"] = pd.Series([None] * len(gdf), dtype="float")

headers = {"accept": "application/json"}

def fetch(index, geometry):
    bpolys = geojson.Feature(geometry=geometry)
    bpolys_collection = geojson.FeatureCollection([bpolys])

    parameters = {
        "topic": topic,
        "bpolys": bpolys_collection,
    }
    for attempt in range(5):
        try:
            print(f"posting request for index {index}")
            startresponse = time.time()
            response = requests.post(url, headers=headers, json=parameters, timeout=600)
            response.raise_for_status()
            result = response.json()
            endresponse = time.time()
            responsetime = endresponse - startresponse
            value = result["result"][0]["result"]["value"]
            return index, value, responsetime
        except requests.RequestException as e:
            print(f"Attempt {attempt + 1} failed at index {index}: {e}")
            if attempt < 3:
                print("Retrying...")
                time.sleep(2)
            else:
                print("Max retries reached. Skipping.")
                return index, None, None

start = time.time()
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(fetch, i, gdf.geometry.iloc[i]) for i in range(len(gdf))]

    for future in as_completed(futures):
        index, value, responsetime = future.result()
        gdf.at[index, "result_value"] = value
        gdf.at[index, "response_time"] = responsetime
        print(f"Completed index {index}: {value}")

end = time.time()
print(f"Calculation took {end - start:.2f} seconds")

gdf.to_file(output_geom_path, driver="GPKG")


posting request for index 0
Completed index 0: 0.8424891262702325
Calculation took 1.47 seconds
